## 1. Setup
This notebook builds a binary sentiment classifier for English movie reviews using the IMDb dataset distributed by TensorFlow/Keras.

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


## 2. Load the IMDb dataset

In [ ]:
MAX_WORDS = 10_000
MAX_LEN = 250

(train_x, train_y), (test_x, test_y) = tf.keras.datasets.imdb.load_data(num_words=MAX_WORDS)
print("Training samples:", len(train_x))
print("Test samples:", len(test_x))
print("Positive fraction (train):", float(np.mean(train_y)))
print("Positive fraction (test):", float(np.mean(test_y)))


## 3. Pad sequences

In [ ]:
x_train = tf.keras.preprocessing.sequence.pad_sequences(train_x, maxlen=MAX_LEN, padding="post", truncating="post")
x_test = tf.keras.preprocessing.sequence.pad_sequences(test_x, maxlen=MAX_LEN, padding="post", truncating="post")
print("Padded training shape:", x_train.shape)
print("Padded test shape:", x_test.shape)


## 4. Define the model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(MAX_LEN,)),
    tf.keras.layers.Embedding(input_dim=MAX_WORDS, output_dim=64, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, dropout=0.2)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="imdb_bilstm")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)
model.summary()


## 5. Train with validation and early stopping

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5, verbose=1),
]
history = model.fit(
    x_train, train_y, validation_split=0.2, epochs=8, batch_size=128,
    callbacks=callbacks, verbose=1,
)


## 6. Plot training history

In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy by Epoch"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss by Epoch"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Binary cross-entropy")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 7. Evaluate once on the held-out test set

In [ ]:
test_loss, test_accuracy, test_auc = model.evaluate(x_test, test_y, verbose=0)
probabilities = model.predict(x_test, batch_size=256, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test ROC-AUC: {test_auc:.4f}")
print("\nClassification report:")
print(classification_report(test_y, predictions, target_names=["Negative", "Positive"], digits=4))


## 8. Confusion matrix

In [ ]:
cm = confusion_matrix(test_y, predictions)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.title("IMDb Test Set Confusion Matrix")
plt.xlabel("Predicted label"); plt.ylabel("True label")
plt.tight_layout(); plt.show()


## 9. Decode review tokens for inspection

In [ ]:
word_index = tf.keras.datasets.imdb.get_word_index()
index_word = {index + 3: word for word, index in word_index.items()}
index_word[0] = "<PAD>"; index_word[1] = "<START>"; index_word[2] = "<UNK>"; index_word[3] = "<UNUSED>"

def decode_review(encoded_review):
    """Convert IMDb integer tokens into a readable, approximate review."""
    return " ".join(index_word.get(int(token), "<UNK>") for token in encoded_review if token != 0)

print(decode_review(train_x[0])[:1000])
print("Label:", "Positive" if train_y[0] == 1 else "Negative")


## 10. Predict sentiment for custom text

In [ ]:
def encode_custom_review(text, max_words=MAX_WORDS, max_len=MAX_LEN):
    """Encode English text using the Keras IMDb word index."""
    words = tf.keras.preprocessing.text.text_to_word_sequence(text)
    encoded = [1]
    for word in words:
        idx = word_index.get(word)
        if idx is None or idx + 3 >= max_words:
            encoded.append(2)
        else:
            encoded.append(idx + 3)
    return tf.keras.preprocessing.sequence.pad_sequences([encoded], maxlen=max_len, padding="post", truncating="post")

def predict_sentiment(text):
    """Predict sentiment and return class confidence and positive probability."""
    encoded = encode_custom_review(text)
    positive_probability = float(model.predict(encoded, verbose=0)[0, 0])
    label = "Positive" if positive_probability >= 0.5 else "Negative"
    confidence = positive_probability if label == "Positive" else 1 - positive_probability
    return {"text": text, "sentiment": label, "confidence": confidence, "positive_probability": positive_probability}

for example in [
    "This movie was wonderful, moving, and beautifully acted.",
    "The story was dull and the acting was disappointing.",
    "I expected more, but some scenes were quite enjoyable.",
]:
    result = predict_sentiment(example)
    print(f"Text: {result['text']}")
    print(f"Prediction: {result['sentiment']} | Confidence: {result['confidence']:.2%} | P(positive): {result['positive_probability']:.2%}\n")


## 11. Save the trained model

In [ ]:
import os
os.makedirs("artifacts", exist_ok=True)
model.save("artifacts/imdb_bilstm.keras")
print("Saved model to artifacts/imdb_bilstm.keras")


## 12. Notes and limitations
- The IMDb dataset uses integer-encoded words from Keras's dataset utilities; custom text is encoded using the same word index.
- The model predicts binary sentiment and is not calibrated for every domain, dialect, or language.
- The displayed test metrics are generated by this notebook when it is run. They should not be copied into project documentation until reproduced in the current environment.